In [ ]:
# imports
import numpy as np
import pandas as pd
from scipy import stats

from dual_pfc_funcs import load_dict, jitter, getParams

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
plt.style.use('scifigs.mplstyle')
SAVE_FIG = False

# parameters
p = getParams()
subjects = p['subjects']
color_map = p['color_map']
plot_sym = p['markers']
data_path = 'preprocessed_data/'

In [ ]:
# get example session noise corr distributions
fname = data_path + 'within_across_rsc_distributions.pkl'
dat = load_dict(fname)
ex_sess = 'Pe180726'
ex_idx = np.where([s == ex_sess for s in dat['SessionNames']])[0][0]

rsc_L = dat['WithinAreaLeftRsc'][ex_idx]
rsc_R = dat['WithinAreaRightRsc'][ex_idx]
rsc_acc = dat['AcrossAreaRsc'][ex_idx]

print("example rsc means:")
print("  within right: {:.3f}".format(np.mean(rsc_R)))
print("  within left: {:.3f}".format(np.mean(rsc_L)))
print("  across: {:.3f}".format(np.mean(rsc_acc)))

# statistics 
alpha = 0.01 
print("example rsc statistics:")
p = stats.ttest_1samp(rsc_R, popmean=0, alternative='greater').pvalue
print("  within right > 0? {}, p = {:.6f}, n = {} pairs".format(p < alpha, p, rsc_R.shape[0]))
p = stats.ttest_1samp(rsc_L, popmean=0, alternative='greater').pvalue
print("  within left > 0? {}, p = {:.6f}, n = {} pairs".format(p < alpha, p, rsc_L.shape[0]))
p = stats.ttest_1samp(rsc_acc, popmean=0, alternative='greater').pvalue
print("  across > 0? {}, p = {:.6f}, n = {} pairs".format(p < alpha, p, rsc_acc.shape[0]))

In [ ]:
# plot results
fig,ax = plt.subplots(3,1, constrained_layout=True, sharex=True, sharey=True)
fig.set_figheight(8)

binsize = 0.025
bins = np.arange(-1,1.001,binsize)
ylim = 0.19
xlim = np.ceil(np.max(np.abs(np.concatenate((rsc_acc,rsc_L,rsc_R)))) / binsize) * binsize

ax[0].hist(rsc_R,bins=bins,color=color_map['within1'],weights=np.ones(len(rsc_R)) / len(rsc_R))
ax[0].plot(rsc_R.mean(),ylim-0.005,color=color_map['within1'],marker='v',markersize=8)
ax[0].set_title('within-area (right)',color=color_map['within1'])

ax[1].hist(rsc_L,bins=bins,color=color_map['within2'],weights=np.ones(len(rsc_L)) / len(rsc_L))
ax[1].plot(rsc_L.mean(),ylim-0.005,color=color_map['within2'],marker='v',markersize=8)
ax[1].set_title('within-area (left)',color=color_map['within2'])

ax[2].hist(rsc_acc,bins=bins,color=color_map['across'],weights=np.ones(len(rsc_acc)) / len(rsc_acc))
ax[2].plot(rsc_acc.mean(),ylim-0.005,color=color_map['across'],marker='v',markersize=8)
ax[2].set_title('across-area',color=color_map['across'])

ax[0].axvline(x=0,color="black",linestyle="--",linewidth=1)
ax[1].axvline(x=0,color="black",linestyle="--",linewidth=1)
ax[2].axvline(x=0,color="black",linestyle="--",linewidth=1)

fig.supxlabel('spike count correlation ($r_{sc}$)')
fig.supylabel('proportion of neuron pairs')
ax[0].set_xlim(-xlim,xlim)
ax[0].set_yticks([0,0.1,0.2])

if SAVE_FIG:
    pdf = PdfPages('figs/rsc_example_sess.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

In [ ]:
# plot mean rsc across sessions
fname = data_path + 'within_across_rsc_means.pkl'
df = pd.read_pickle(fname,compression='gzip')

# plot box and whisker of session means
fig,ax = plt.subplots(3,1, constrained_layout=True, sharex=True, sharey=True)
fig.set_figheight(8)
y=0.5

# within-area right
box = ax[0].boxplot(df['WithinAreaRightRsc'], patch_artist=True, positions=[y],
                  notch=False, vert=0, widths=0.2, showmeans=True, meanline=True, showfliers=False, medianprops={'linewidth':0}, meanprops={'color':'black'})
box['boxes'][0].set_facecolor(color_map['within1'])
ax[0].set_ylabel('within-area (right)',color=color_map['within1'])
ax[0].spines['left'].set_visible(False)

# within-area left
box = ax[1].boxplot(df['WithinAreaLeftRsc'], patch_artist=True, positions=[y],
                  notch=False, vert=0, widths=0.2, showmeans=True, meanline=True, showfliers=False, medianprops={'linewidth':0}, meanprops={'color':'black'})
box['boxes'][0].set_facecolor(color_map['within2'])
ax[1].set_ylabel('within-area (left)',color=color_map['within2'])
ax[1].spines['left'].set_visible(False)

# across-area
box = ax[2].boxplot(df['AcrossAreaRsc'], patch_artist=True, positions=[y],
                  notch=False, vert=0, widths=0.2, showmeans=True, meanline=True, showfliers=False, medianprops={'linewidth':0}, meanprops={'color':'black'})
box['boxes'][0].set_facecolor(color_map['across'])
ax[2].set_ylabel('across-area',color=color_map['across'])
ax[2].spines['left'].set_visible(False)

# add individual data points
for j, subject in enumerate(subjects):
    mask = df['Subject']==subject
    right_curr = df[mask]['WithinAreaRightRsc']
    left_curr = df[mask]['WithinAreaLeftRsc']
    across_curr = df[mask]['AcrossAreaRsc']
    
    j1 = y+.23+jitter(len(right_curr),spacing=0.05,rand_seed=j)
    j2 = y+.23+jitter(len(left_curr),spacing=0.05,rand_seed=j+1)
    j3 = y+.23+jitter(len(across_curr),spacing=0.05,rand_seed=j+10)
    ax[0].scatter(right_curr, j1, color=color_map['within1'], alpha=0.5,s=12,marker=plot_sym[j],edgecolors='none')
    ax[1].scatter(left_curr, j2, color=color_map['within2'], alpha=0.5,s=12,marker=plot_sym[j],edgecolors='none')
    ax[2].scatter(across_curr, j3, color=color_map['across'], alpha=0.5,s=12,marker=plot_sym[j],edgecolors='none',label='Monkey {:s}'.format(subject[:2].title()))
    
    if subject == 'pepe':
        # mark example session
        ax[0].scatter(right_curr[ex_idx],j1[ex_idx], color=color_map['within1'],alpha=1,s=40,marker="*",edgecolors='black',linewidths=0.5)
        ax[1].scatter(left_curr[ex_idx],j2[ex_idx], color=color_map['within2'],alpha=1,s=40,marker="*",edgecolors='black',linewidths=0.5)
        ax[2].scatter(across_curr[ex_idx],j3[ex_idx], color=color_map['across'],alpha=1,s=40,marker="*",edgecolors='black',linewidths=0.5)

ax[0].axvline(x=0,color="black",linestyle="--",linewidth=1,alpha=0.5)
ax[1].axvline(x=0,color="black",linestyle="--",linewidth=1,alpha=0.5)
ax[2].axvline(x=0,color="black",linestyle="--",linewidth=1,alpha=0.5)

# formatting
plt.yticks([])
plt.xticks([0,0.025,0.05])
plt.xlabel('mean $r_{sc}$', fontsize=12)
plt.legend(loc='lower right',markerscale=1.5)

if SAVE_FIG:
    pdf = PdfPages('figs/rsc_all_sess.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

In [ ]:
# statistics 
alpha = 0.01 
print("rsc statistics across sessions:")
p = stats.ttest_rel(a=df['WithinAreaRightRsc'], b=df['AcrossAreaRsc'],alternative='greater').pvalue
print("  within right > across? {}, p = {:.6f}".format(p < alpha, p))
p = stats.ttest_rel(a=df['WithinAreaLeftRsc'], b=df['AcrossAreaRsc'],alternative='greater').pvalue
print("  within left > across? {}, p = {:.6f}".format(p < alpha, p))


print()
for sub in subjects:
    filt = df['Subject'] == sub
    tmp_df = df[filt]

    print('Subject: {}'.format(sub))

    # test if across and within are different for this monkey
    _,pright = stats.ttest_rel(tmp_df['WithinAreaRightRsc'],tmp_df['AcrossAreaRsc'],alternative='greater')
    _,pleft = stats.ttest_rel(tmp_df['WithinAreaLeftRsc'],tmp_df['AcrossAreaRsc'],alternative='greater')
    print('  within right > across? {}, p = {:5f}'.format(pright<alpha,pright))
    print('  within left > across? {}, p = {:5f}'.format(pleft<alpha,pleft))